# Part 6 – Final Evaluation & Metrics

**Branch:** `feature/test-evaluation-metrics`


## 14. Final Evaluation (TEST split)



### 14a. Confusion matrix

In [ ]:
test_summary = pairs_summary_df[pairs_summary_df['split'] == 'test']
y_test = test_summary['true_change']
test_pred = test_summary['predicted_change']
labels_sorted = CHANGE_LABELS


mask = test_pred != 'None'
y_test_cm = y_test[mask]
test_pred_cm = test_pred[mask]

cm = confusion_matrix(y_test_cm, test_pred_cm, labels=CHANGE_LABELS)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CHANGE_LABELS, yticklabels=CHANGE_LABELS)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Rule-Based Change Detection -- Confusion Matrix (TEST split, n={len(y_test_cm)})')
plt.tight_layout()
plt.savefig('/content/change_confusion_matrix_rules.png', dpi=150)
plt.show()

test_report = classification_report(y_test, test_pred, labels=CHANGE_LABELS,
                                    output_dict=True, zero_division=0)
test_metrics = pair_metrics(test_summary)
test_report['accuracy'] = test_metrics['accuracy']
print(classification_report(y_test, test_pred, labels=CHANGE_LABELS, zero_division=0))

answered = test_pred != 'None'
print(f"Accuracy over ALL {len(y_test)} test pairs ('None' = wrong): {test_metrics['accuracy']:.3f}")
print(f"Coverage (pairs where a change was reported):   {test_metrics['coverage']:.3f}")
print(f"Accuracy on the answered pairs only:            {(y_test[answered] == test_pred[answered]).mean():.3f}")
print(f"Change type AND object both correct:            {test_metrics['type_and_object_acc']:.3f}")

### 14b. Precision / recall / F1 per class

In [ ]:
report_df = pd.DataFrame(test_report).transpose().loc[labels_sorted, ['precision', 'recall', 'f1-score']]

plt.figure(figsize=(6, 4))
sns.heatmap(report_df, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title('Rule-Based Matching -- Precision / Recall / F1 by Class (TEST split)')
plt.tight_layout()
plt.savefig('/content/classification_report_heatmap.png', dpi=150)
plt.show()


### 14c. Events per pair (multi-object detections)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
counts = pairs_summary_df['n_events'].value_counts().sort_index()
bars = ax.bar(counts.index.astype(str), counts.values, color='#4C72B0')
ax.bar_label(bars, padding=3)
ax.set_xlabel('Change events detected in the pair')
ax.set_ylabel('Number of pairs')
ax.set_title('How many simultaneous change events the matcher found per pair')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/events_per_pair.png', dpi=150)
plt.show()


### 14d. Prediction distribution and errors

In [ ]:
print("True change distribution by split:")
print(pairs_summary_df.groupby('split')['true_change'].value_counts().unstack(fill_value=0))
print("\nPredicted (primary-event) change distribution by split:")
print(pairs_summary_df.groupby('split')['predicted_change'].value_counts().unstack(fill_value=0))


In [ ]:
test_rows = pairs_summary_df[pairs_summary_df['split'] == 'test']
wrong = test_rows[test_rows['true_change'] != test_rows['predicted_change']]
print(f"{len(wrong)} / {len(test_rows)} TEST pairs misclassified (primary-event vs. ground truth)")
wrong[['pair_id', 'true_change', 'predicted_change', 'predicted_change_prob',
       'true_object', 'predicted_object', 'n_events', 'n_final_changes']].head(15)


### 14e. Overall system performance

In [ ]:
system_summary = pd.DataFrame([
    {'component': 'Detector\n(YOLO11s)', 'metric': 'mAP50',    'score': yolo_test_metrics['mAP50']},
    {'component': 'Detector\n(YOLO11s)', 'metric': 'mAP50-95', 'score': yolo_test_metrics['mAP50-95']},
    {'component': 'Change\nMatching',    'metric': 'Accuracy', 'score': test_report['accuracy']},
    {'component': 'Change\nMatching',    'metric': 'Macro F1', 'score': test_report['macro avg']['f1-score']},
])
colors_map = {'mAP50': '#4C72B0', 'mAP50-95': '#8CA9D6', 'Accuracy': '#DD8452', 'Macro F1': '#F0B27A'}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(system_summary['component'] + '\n' + system_summary['metric'], system_summary['score'],
              color=[colors_map[m] for m in system_summary['metric']])
for b, v in zip(bars, system_summary['score']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Overall System Performance -- TEST Split')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/system_performance_summary.png', dpi=150)
plt.show()


## 15. End-to-End Pipeline Trace (your demo pair)

What each stage receives and returns, all for the single reference/query pair you uploaded at the top:

1. **Raw input** — your uploaded reference / query photos.
2. **Preprocessed input** — the same pair after CLAHE; this is what YOLO sees.
3. **YOLO output** — `{class, bbox, confidence}` per box, for reference and query.
4. **Matching input** — the two box lists, homography `H`, thresholds, appearance histograms, pixel change map.
5. **Pixel-change evidence** — where the aligned pair actually differs (used to verify every candidate).
6. **Matching output** — `matched_pairs` and `events`, each event with its change-probability.
7. **Final decision** — the events that clear the threshold.

In [ ]:
def to_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def draw_yolo(img_bgr, boxes):
    out = img_bgr.copy()
    for b in boxes:
        draw_labeled_box(out, *b['bbox'], f"{b['class']} {b['confidence']:.2f}", (0, 0, 255))
    return out

# ---- Stages 1-3: raw, preprocessed and YOLO-output images (reusing what was already computed above) ----
stages = [
    ("Stage 1 -- Raw input", cv2.imread(USER_REFERENCE_PATH), cv2.imread(USER_QUERY_PATH)),
    ("Stage 2 -- Preprocessed input to YOLO (CLAHE)", demo_reference_pre, demo_query_pre),
    ("Stage 3 -- YOLO output", draw_yolo(demo_reference_pre, demo_reference_boxes), draw_yolo(demo_query_pre, demo_query_boxes)),
]
fig, axes = plt.subplots(len(stages), 2, figsize=(11, 4.6 * len(stages)))
for r, (label, b_img, a_img) in enumerate(stages):
    axes[r, 0].imshow(to_rgb(b_img)); axes[r, 0].axis('off'); axes[r, 0].set_title(f"{label} -- Reference", fontsize=11)
    axes[r, 1].imshow(to_rgb(a_img)); axes[r, 1].axis('off'); axes[r, 1].set_title(f"{label} -- Query", fontsize=11)
plt.suptitle("System Input and Output -- Pipeline Trace (your uploaded pair)", fontsize=15, y=1.0)
plt.tight_layout()
plt.savefig('/content/system_input_output_trace_user.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Stage 3 -- YOLO output: reference_boxes ({len(demo_reference_boxes)}), query_boxes ({len(demo_query_boxes)})")
display(boxes_df(demo_reference_boxes))
display(boxes_df(demo_query_boxes))

# ---- Stage 4: matching input ----
print("\nStage 4 -- Matching input")
print("H (reference -> query):")
print(np.array2string(demo_H, precision=4, suppress_small=True))
print(f"camera_shift_residual = {demo_residual_px:.2f}px  |  base move_thresh = {demo_move_thresh:.1f}px  |  "
      f"image_diag = {demo_image_diag:.1f}px")
print(f"pixel_k = {FINAL_PARAMS['pixel_k']}  |  visual_sat = {FINAL_PARAMS['visual_sat']}  |  "
      f"min_box_change = {FINAL_PARAMS['min_box_change']}  |  ghost_iou = {FINAL_PARAMS['ghost_iou']}")
print(f"reference_img = pixel array {demo_reference_pre.shape}  |  query_img = pixel array {demo_query_pre.shape}")

# ---- Stage 5: pixel-change evidence ----
demo_change = demo_result['change']
if demo_change is not None:
    k = FINAL_PARAMS['pixel_k']
    mask = changed_mask(demo_change, k)
    s = demo_change['scale']
    qry_small = cv2.resize(demo_query_bgr, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_AREA)
    overlay = qry_small.copy()
    overlay[mask] = (0.45 * overlay[mask] + 0.55 * np.array([0, 0, 255])).astype(np.uint8)
    colours = {'Add': (0, 180, 0), 'Delete': (0, 0, 255), 'Move': (0, 140, 255)}
    for e in demo_decision['changes']:
        for key, H_box in (('bbox_ref', demo_H), ('bbox_qry', None)):
            if key in e:
                x1, y1, x2, y2 = [int(round(v * s)) for v in (warp_bbox(e[key], H_box) if H_box is not None else e[key])]
                cv2.rectangle(overlay, (x1, y1), (x2, y2), colours[e['type']], 2)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    axes[0].imshow(np.clip(demo_change['score'].astype(np.float32) / (3 * k), 0, 1), cmap='inferno')
    axes[0].set_title('Stage 5 -- change score (robust z, reference warped onto query)', fontsize=10)
    axes[1].imshow(to_rgb(overlay))
    axes[1].set_title(f'Changed pixels (z > {k}) + final events', fontsize=10)
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('/content/demo_change_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Stage 5 -- {mask.mean() * 100:.1f}% of the comparable image area changed.")

# ---- Stage 6: matching output -- every CANDIDATE event with its change-probability ----
matched_df, candidate_events_df, suppressed_df = matching_output_tables(demo_ctx)
print(f"\nStage 6 -- Matching output: matched_pairs ({len(matched_df)}), candidate events ({len(demo_events)}), "
      f"cross-class pairs suppressed as noise ({len(demo_suppressed)})")
display(matched_df)
print("\nAll candidate change events, ranked by change-probability:")
display(candidate_events_df)
if len(suppressed_df):
    print("\nSuppressed as likely detector class-flip noise (not reported):")
    display(suppressed_df)

# ---- Stage 7: FINAL DECISION -- the events that actually cleared the threshold ----
print(f"\nStage 7 -- Final decision (threshold = {FINAL_PARAMS['prob_threshold']*100:.0f}%, {demo_ctx['pass']} pass)")
print(demo_decision['summary'])
if demo_decision['has_change']:
    final_decision_df = pd.DataFrame(
        [{'object': e['object'], 'change_type': e['type'], 'probability_pct': round(e['probability'] * 100, 1)}
         for e in demo_decision['changes']],
        columns=['object', 'change_type', 'probability_pct'])
    display(final_decision_df)